# Building a QA Agent with Gemini

We will build a basic Question Answering (QA) agent using Google's Gemini API.
We will start by directly calling Gemini to analyze a PDF document containing health insurance policy details.
Then, we will refactor this logic into a reusable Python class, laying the groundwork for the next notebook where we will wrap this agent in an Agent2Agent (A2A) server.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/A2A-MCP

/content/drive/MyDrive/A2A-MCP


In [3]:
%%writefile requirements.txt
google-adk[a2a]==1.19.0
a2a-sdk[http-server]==0.3.16
mcp==1.19.0
agent-framework==1.0.0b251120
agent-framework-a2a==1.0.0b251120
langchain-mcp-adapters
langgraph
langchain[openai]
google-auth==2.42.0
httpx==0.28.1
ipython==8.32.0
nest-asyncio==1.6.0
python-dotenv==1.0.1
requests>=2.32.4,<3.0.0
uvicorn==0.34.0

Overwriting requirements.txt


In [4]:
%pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 24.6 MB/s eta 0:00:0000:0100:01m


In [5]:
!uv pip install --system -r requirements.txt

Using Python 3.12.13 environment at: /usr
Resolved 181 packages in 6.35s                                       
Prepared 55 packages in 1.29s                                            
Uninstalled 19 packages in 572ms
Installed 55 packages in 188ms                              
 + a2a-sdk==0.3.16
 + agent-framework==1.0.0b251120
 + agent-framework-a2a==1.0.0b251120
 + agent-framework-core==1.0.0b251001
 + asgiref==3.11.1
 + asttokens==3.0.1
 + azure-core==1.39.0
 + azure-core-tracing-opentelemetry==1.0.0b12
 + azure-identity==1.26.0b2
 + azure-monitor-opentelemetry==1.8.2
 + azure-monitor-opentelemetry-exporter==1.0.0b45
 + executing==2.2.1
 - fastapi==0.135.3
 + fastapi==0.118.3
 - google-adk==1.29.0
 + google-adk==1.19.0
 - google-auth==2.47.0
 + google-auth==2.42.0
 - google-cloud-aiplatform==1.147.0
 + google-cloud-aiplatform==1.130.0
 - google-genai==1.68.0
 + google-genai==1.55.0
 - ipython==7.34.0
 + ipython==8.32.0
 + isodate==0.7.2
 + jedi==0.19.2
 + langchain-mcp-adapters==0

## 3.1. Import Libraries and Setup

First, import the necessary libraries. We will use the Gemini REST API so the notebook only needs a Gemini API key and standard Python libraries.

In [ ]:
import base64
from pathlib import Path
import requests
from IPython.display import Markdown, display

from helpers import get_gemini_api_key, get_gemini_model, setup_env

In [7]:
setup_env()

For this workshop, the `.env` file only needs a Gemini API key and optional Gemini settings such as `GEMINI_MODEL`.

## 3.2. Initialize the Client and Load Data

Here you initialize the Gemini request settings using the API key from `.env`.


In [8]:
api_key = get_gemini_api_key()
model = get_gemini_model()

In [9]:
api_url = (
    "https://generativelanguage.googleapis.com/v1beta/models/"
    f"{model}:generateContent"
)

You also read the insurance policy PDF (`2026AnthemgHIPSBC.pdf`) and encode it in base64 so it can be passed to the model as context. You can find the PDF document in the `data` folder.

In [10]:
with Path("./data/2026AnthemgHIPSBC.pdf").open("rb") as file:
    pdf_data = base64.standard_b64encode(file.read()).decode("utf-8")

## 3.3. Query the Model

Now you will send a specific query to the model: "How much would I pay for mental health therapy?". You provide Gemini with a system instruction to act as an expert insurance agent and pass the PDF document alongside the user's text prompt.

In [11]:
prompt = "How much would I pay for mental health therapy?"

In [12]:
response = requests.post(
    api_url,
    headers={
        "Content-Type": "application/json",
        "x-goog-api-key": api_key,
    },
    json={
        "system_instruction": {
            "parts": [
                {
                    "text": "You are an expert insurance agent designed to assist with coverage queries. Use the provided documents to answer questions about insurance policies. If the information is not available in the documents, respond with 'I don't know'."
                }
            ]
        },
        "generationConfig": {"temperature": 0.2},
        "contents": [
            {
                "parts": [
                    {
                        "inline_data": {
                            "mime_type": "application/pdf",
                            "data": pdf_data,
                        }
                    },
                    {"text": prompt},
                ]
            }
        ],
    },
    timeout=120,
)
response.raise_for_status()
result = response.json()

In [13]:
response_text = result["candidates"][0]["content"]["parts"][0]["text"].replace("$", r"\\$")
display(Markdown(response_text))

If you need mental health, behavioral health, or substance abuse services, you would pay 10% coinsurance for in-network outpatient office visits and 30% coinsurance for out-of-network outpatient office visits. For inpatient services, you would pay 10% coinsurance for in-network and 30% coinsurance for out-of-network.

## 3.4. Refactor into an Agent Class

To make this code reusable and easier to integrate into an A2A server later, you will wrap the logic into a `PolicyAgent` class in a file named `agents.py`. This class initializes the client and data in the `__init__` method and exposes an `answer_query` method.

In [14]:
%%writefile agents.py
import base64
from pathlib import Path

import requests

from helpers import get_gemini_api_key, get_gemini_model, setup_env

SYSTEM_INSTRUCTION = (
    "You are an expert insurance agent designed to assist with coverage "
    "queries. Use the provided documents to answer questions about insurance "
    "policies. If the information is not available in the documents, respond "
    "with 'I don't know'."
)

class PolicyAgent:
    def __init__(self) -> None:
        setup_env()
        self.api_key = get_gemini_api_key()
        self.model = get_gemini_model()
        self.api_url = (
            "https://generativelanguage.googleapis.com/v1beta/models/"
            f"{self.model}:generateContent"
        )
        with Path("./data/2026AnthemgHIPSBC.pdf").open("rb") as file:
            self.pdf_data = base64.standard_b64encode(file.read()).decode("utf-8")

    def answer_query(self, prompt: str) -> str:
        response = requests.post(
            self.api_url,
            headers={
                "Content-Type": "application/json",
                "x-goog-api-key": self.api_key,
            },
            json={
                "system_instruction": {
                    "parts": [{"text": SYSTEM_INSTRUCTION}]
                },
                "generationConfig": {"temperature": 0.2},
                "contents": [
                    {
                        "parts": [
                            {
                                "inline_data": {
                                    "mime_type": "application/pdf",
                                    "data": self.pdf_data,
                                }
                            },
                            {"text": prompt},
                        ]
                    }
                ],
            },
            timeout=120,
        )
        response.raise_for_status()
        result = response.json()
        return result["candidates"][0]["content"]["parts"][0]["text"].replace("$", r"\\$")

Overwriting agents.py


## 3.5. Test the Agent Class

Finally, import the `PolicyAgent` class you just created and test it with the same query to ensure it works as expected.

In [15]:
from agents import PolicyAgent

print("Running Health Insurance Policy Agent")
agent = PolicyAgent()
prompt = "How much would I pay for mental health therapy?"

response = agent.answer_query(prompt)
display(Markdown(response))

Running Health Insurance Policy Agent


If you need mental health, behavioral health, or substance abuse services:

*   **In-Network Provider:**
    *   Office Visit: 10% coinsurance
    *   Other Outpatient: 10% coinsurance
    *   Inpatient services: 10% coinsurance
*   **Out-of-Network Provider:**
    *   Office Visit: 30% coinsurance
    *   Other Outpatient: 30% coinsurance
    *   Inpatient services: 30% coinsurance

## 3.6. Resources

- [Gemini API Document Understanding](https://ai.google.dev/gemini-api/docs/document-processing)
- [Gemini API Models](https://ai.google.dev/gemini-api/docs/models)